# Diabetes Prediction Model (NHANES 2017–2018)

**Goal:** a screening model that flags adults likely to have diabetes using **only physical exam measurements** (body measures, blood pressure, pulse) plus age and sex. **No blood test of any kind** is needed to use it.

**Pipeline (one section per step):**
1. **Feature identification**: define the target and cohort, decide which columns are allowed as features ← *this section*
2. Feature table build
3. Exploratory data analysis (EDA)
4. Feature preparation and engineering
5. Base model
6. Model fitting
7. Metrics
8. Recalibration and optimization
9. Final evaluation

Run the cells top to bottom. Each cell does one job, described in the markdown cell above it.

## Step 1 — Feature identification

Three questions, answered in this order:
1. **Target:** what exactly are we predicting?
2. **Cohort:** whose data do we learn from?
3. **Features:** which columns may the model see?

Two rules decide the features:
- **Scope rule:** only physical exam measurements, plus age and sex.
- **Leakage rule:** a feature is allowed only if it would be known **before** we know the answer, at the moment we want to predict.

Blood tests (HbA1c, questionnaire answers) are still used to **define the target**; they are just never shown to the model.

### 1.1 Imports, paths and constants

Loads the libraries, points to the `Data/` folder, and sets every number the later cells rely on in one place, so nothing is hard-coded further down.

NHANES stores answers as numeric codes. For yes/no questions, `1` = yes, `2` = no, `7` = refused, `9` = don't know.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../Data")

MIN_AGE = 20                 # adults only
HBA1C_DIABETES_CUTOFF = 6.5  # %, the clinical threshold for diabetes
YES = 1                      # NHANES code for "yes"

pd.set_option("display.max_columns", 50)

### 1.2 Load the raw tables

Reads the five NHANES tables this project needs into a dictionary called `raw`, keyed by a short name, and prints the size of each one. Nothing is changed or joined yet.

- `demographics`, `body`, `blood_pressure`: sources of the features (and the pregnancy filter)
- `questionnaire`, `hba1c`: used only to build the target and the cohort filters

In [2]:
TABLE_FILES = {
    "demographics": "demographics.csv",
    "body": "body_measures.csv",
    "blood_pressure": "blood_pressure.csv",
    "questionnaire": "diabetes_questionnaire.csv",
    "hba1c": "hba1c.csv",
}

raw = {name: pd.read_csv(DATA_DIR / file) for name, file in TABLE_FILES.items()}

pd.DataFrame(
    {name: {"rows": len(df), "columns": df.shape[1]} for name, df in raw.items()}
).T

,rows,columns
demographics,9254,46
body,8704,21
blood_pressure,8704,21
questionnaire,8897,54
hba1c,6401,2


### 1.3 Check the join key

Every table identifies a person by `person_id`. Before joining, this cell confirms each table has **at most one row per person**. A duplicated id would silently copy that person's rows during the join.

In [3]:
for name, df in raw.items():
    assert df["person_id"].is_unique, f"{name} has duplicate person_id values"

print("All tables have one row per person_id.")

All tables have one row per person_id.


### 1.4 Join the tables into one wide table

Starts from `demographics` (everyone in the survey) and **left-joins** every other table on `person_id`. A left join keeps every person, even if they have no row in a table; their missing measurements become `NaN`.

In [4]:
merged = raw["demographics"]
for name, df in raw.items():
    if name != "demographics":
        merged = merged.merge(df, on="person_id", how="left")
merged = merged.copy()  # repeated joins leave pandas' internal storage fragmented; copying tidies it

print(f"{merged.shape[0]:,} people x {merged.shape[1]} columns")

9,254 people x 140 columns


### 1.5 Define the target

A person counts as **diabetic** (`diabetes = 1`) if **either**:
- a doctor has told them they have diabetes (`told_diabetes == 1`), **or**
- their HbA1c is at or above 6.5%, which catches people with **undiagnosed** diabetes.

Answers of *borderline* (`3`) or *don't know* (`9`) count as diabetic only if their HbA1c is 6.5% or higher.

Because these columns define the answer, they can **never** be features.

In [5]:
def make_target(df):
    told = df["told_diabetes"] == YES
    high_a1c = df["hba1c_percent"] >= HBA1C_DIABETES_CUTOFF
    return (told | high_a1c).astype(int)


merged["diabetes"] = make_target(merged)

### 1.6 Apply the cohort filters

Each filter is a named rule. They are applied in order, and the table printed afterward (the **cohort funnel**) shows how many people and diabetic cases each rule leaves. This makes every exclusion visible and easy to question.

| Filter | Why |
|---|---|
| Adult (20+) | Diabetes in children is mostly type 1, a different disease |
| Has an HbA1c result | Without it the target can't be checked for undiagnosed diabetes |
| Not pregnant | Pregnancy lowers HbA1c and changes weight and blood pressure |
| Not taking insulin | Insulin users have advanced, already-treated diabetes; the screening population isn't on insulin |

In [6]:
COHORT_FILTERS = {
    f"age >= {MIN_AGE}": lambda df: df["age_years"] >= MIN_AGE,
    "has HbA1c result": lambda df: df["hba1c_percent"].notna(),
    "not pregnant": lambda df: df["pregnant_at_exam"] != YES,
    "not taking insulin": lambda df: df["taking_insulin"] != YES,
}

cohort = merged
funnel = [{"step": "all survey participants", "people": len(cohort), "diabetic": cohort["diabetes"].sum()}]
for step, rule in COHORT_FILTERS.items():
    cohort = cohort[rule(cohort)]
    funnel.append({"step": step, "people": len(cohort), "diabetic": cohort["diabetes"].sum()})

funnel = pd.DataFrame(funnel)
funnel["diabetic_rate"] = (funnel["diabetic"] / funnel["people"]).round(3)
funnel

,step,people,diabetic,diabetic_rate
0,all survey participants,9254,1085,0.117
1,age >= 20,5569,1066,0.191
2,has HbA1c result,5018,994,0.198
3,not pregnant,4971,994,0.200
4,not taking insulin,4742,766,0.162


### 1.7 Break down the diabetic cases

Splits the cohort's diabetic cases into **diagnosed** (a doctor told them) and **undiagnosed** (HbA1c ≥ 6.5% but never told). The undiagnosed group is who a screening model exists to find.

In [7]:
told = cohort["told_diabetes"] == YES
undiagnosed = (cohort["hba1c_percent"] >= HBA1C_DIABETES_CUTOFF) & ~told

pd.Series({
    "diagnosed": told.sum(),
    "undiagnosed": undiagnosed.sum(),
    "not diabetic": (cohort["diabetes"] == 0).sum(),
})

diagnosed        577
undiagnosed      189
not diabetic    3976
dtype: int64

### 1.8 Feature catalog: every candidate column and its decision

Lists the candidate columns with a **group**, a **decision** (`use` or `drop`) and the **reason**. This table is the written record of Step 1; later steps only build features from columns marked `use`.

Blood pressure keeps readings 1–3 for now. Combining them into one average is a feature engineering job (Step 4). Reading 4 is dropped: NHANES only takes it when an earlier reading failed, so it is missing for over 90% of people.

In [8]:
catalog = pd.DataFrame(
    [
        # column, group, decision, reason
        ("age_years", "age and sex", "use", "Major diabetes risk factor; always known"),
        ("gender", "age and sex", "use", "Risk and body shape differ by sex; always known"),
        ("bmi", "body", "use", "Strongest known risk factor"),
        ("waist_cm", "body", "use", "Belly fat; adds signal beyond BMI"),
        ("hip_cm", "body", "use", "Gives waist-to-hip ratio in Step 4"),
        ("weight_kg", "body", "use", "Raw body size"),
        ("height_cm", "body", "use", "Raw body size; gives waist-to-height ratio in Step 4"),
        ("arm_circumference_cm", "body", "use", "Muscle and fat on the arm"),
        ("upper_leg_length_cm", "body", "use", "Shorter legs relative to height are linked to diabetes risk"),
        ("upper_arm_length_cm", "body", "use", "Body proportion measure"),
        ("systolic_bp_1", "blood pressure", "use", "Reading 1 of 3; averaged in Step 4"),
        ("systolic_bp_2", "blood pressure", "use", "Reading 2 of 3"),
        ("systolic_bp_3", "blood pressure", "use", "Reading 3 of 3"),
        ("diastolic_bp_1", "blood pressure", "use", "Reading 1 of 3; averaged in Step 4"),
        ("diastolic_bp_2", "blood pressure", "use", "Reading 2 of 3"),
        ("diastolic_bp_3", "blood pressure", "use", "Reading 3 of 3"),
        ("pulse_60s", "pulse", "use", "Higher resting heart rate is linked to diabetes"),
        ("systolic_bp_4", "blood pressure", "drop", "Only taken when an earlier reading failed; mostly missing"),
        ("diastolic_bp_4", "blood pressure", "drop", "Only taken when an earlier reading failed; mostly missing"),
        ("recumbent_length_cm", "body", "drop", "Measured only for infants"),
        ("head_circumference_cm", "body", "drop", "Measured only for infants"),
        ("hba1c_percent", "target-defining", "drop", "Defines the target (direct leakage)"),
        ("told_diabetes", "target-defining", "drop", "Defines the target (direct leakage)"),
        ("taking_insulin", "cohort filter", "drop", "Used to filter the cohort; constant afterward"),
        ("pregnant_at_exam", "cohort filter", "drop", "Used to filter the cohort"),
        ("taking_diabetes_pills", "diabetes follow-up", "drop", "Only diagnosed people take them (treatment leakage)"),
        ("diabetic_retinopathy", "diabetes follow-up", "drop", "A complication of diabetes, so it comes after the answer"),
        ("race_ethnicity_with_asian", "demographics", "drop", "Not a physical measurement (out of scope)"),
        ("education_adult", "demographics", "drop", "Not a physical measurement (out of scope)"),
        ("income_to_poverty_ratio", "demographics", "drop", "Not a physical measurement (out of scope)"),
        ("design_psu", "survey design", "drop", "Survey sampling detail, not a property of the person"),
        ("exam_weight", "survey design", "drop", "Survey weight, not a property of the person"),
    ],
    columns=["column", "group", "decision", "reason"],
)

catalog

,column,group,decision,reason
0,age_years,age and sex,use,Major diabetes risk factor; always known
1,gender,age and sex,use,Risk and body shape differ by sex; always known
2,bmi,body,use,Strongest known risk factor
3,waist_cm,body,use,Belly fat; adds signal beyond BMI
4,hip_cm,body,use,Gives waist-to-hip ratio in Step 4
5,weight_kg,body,use,Raw body size
6,height_cm,body,use,Raw body size; gives waist-to-height ratio in ...
7,arm_circumference_cm,body,use,Muscle and fat on the arm
8,upper_leg_length_cm,body,use,Shorter legs relative to height are linked to ...
9,upper_arm_length_cm,body,use,Body proportion measure


### 1.9 Collect the selected features and the target

Turns the catalog into the two names the rest of the notebook uses:
- `FEATURES`: the columns marked `use`
- `TARGET`: the column the model predicts

In [9]:
FEATURES = catalog.loc[catalog["decision"] == "use", "column"].tolist()
TARGET = "diabetes"

print(f"{len(FEATURES)} features selected")
catalog[catalog["decision"] == "use"].groupby("group", sort=False)["column"].apply(list)

17 features selected


group
age and sex                                     [age_years, gender]
body              [bmi, waist_cm, hip_cm, weight_kg, height_cm, ...
blood pressure    [systolic_bp_1, systolic_bp_2, systolic_bp_3, ...
pulse                                                   [pulse_60s]
Name: column, dtype: object

### 1.10 Leakage guard

A safety check that fails loudly if a forbidden column ever ends up in `FEATURES`, for example after someone edits the catalog by mistake. It also confirms every selected column really exists in the data.

In [10]:
FORBIDDEN = set(catalog.loc[catalog["decision"] == "drop", "column"]) | {TARGET}

leaks = FORBIDDEN & set(FEATURES)
assert not leaks, f"Leaking columns in FEATURES: {leaks}"

missing = set(FEATURES) - set(cohort.columns)
assert not missing, f"Columns not found in the data: {missing}"

print("Leakage guard passed: no forbidden columns, all features exist.")

Leakage guard passed: no forbidden columns, all features exist.


### 1.11 Preview the final table

Builds `final_df`: one row per person in the cohort, with `person_id`, the selected `FEATURES` and the `TARGET` column, and nothing else. Shows the first 5 rows as a last check that Step 1 produced what we expect.

In [11]:
final_df = cohort[["person_id"] + FEATURES + [TARGET]].reset_index(drop=True)

print(f"{final_df.shape[0]:,} people x {final_df.shape[1]} columns")
final_df.head(5)

4,742 people x 19 columns


,person_id,age_years,gender,bmi,waist_cm,hip_cm,weight_kg,height_cm,arm_circumference_cm,upper_leg_length_cm,upper_arm_length_cm,systolic_bp_1,systolic_bp_2,systolic_bp_3,diastolic_bp_1,diastolic_bp_2,diastolic_bp_3,pulse_60s,diabetes
0,93705,66.0,2.0,31.7,101.8,110.0,79.5,158.3,32.0,37.0,36.0,NaN,NaN,202.0,NaN,NaN,62.0,52.0,0
1,93708,66.0,2.0,23.7,88.2,90.1,53.5,150.2,27.4,31.8,30.6,NaN,138.0,144.0,NaN,78.0,76.0,68.0,0
2,93709,75.0,2.0,38.9,113.0,131.3,88.8,151.1,38.1,31.9,36.0,120.0,118.0,118.0,66.0,66.0,68.0,74.0,0
3,93711,56.0,1.0,21.3,86.6,90.7,62.1,170.6,30.8,40.1,37.2,108.0,94.0,102.0,68.0,68.0,64.0,62.0,0
4,93713,67.0,1.0,23.5,99.7,91.0,74.9,178.6,30.6,39.1,41.4,104.0,104.0,106.0,70.0,74.0,72.0,90.0,0


### Step 1 summary

- **Target:** diagnosed diabetes **or** HbA1c ≥ 6.5%.
- **Cohort:** adults (20+) with an HbA1c result, not pregnant and not taking insulin. See the funnel in 1.6.
- **Features:** age, sex, body measures, three blood pressure readings and pulse. See 1.9. No blood test is needed to use the model.
- **Missing values are kept for now:** some people lack a body measure or a blood pressure reading. How to handle them is decided in EDA and feature preparation.
- **Excluded:** everything that defines the target, all blood tests, anything that only exists *because* someone was diagnosed, and non-physical demographics.

**Next, Step 2 (feature table build):** keep only `person_id`, `FEATURES` and `TARGET` for the cohort, and save that table.